In [27]:
import pickle

import pandas as pd
import numpy as np
import sklearn

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

In [28]:
print(f'pandas=={pd.__version__}')
print(f'numpy=={np.__version__}')
print(f'sklearn=={sklearn.__version__}')

pandas==2.1.2
numpy==1.26.1
sklearn==1.1.0


In [29]:
### Training on the final model

def load_data():
    data_url = 'cleaned_property_data.csv'
    df = pd.read_csv(data_url)
    return df

In [42]:

def train_model(df):
    numerical = ['total_num_rooms','gross_area', 'adjusted_tax', 'bed_rms', 'full_bth', 
                          'kitchens', 
                         'num_parking', 'floor_number', 
                          'building_age', 'remodel_age'] 

    categorical = [
                            'land_use_description_rank',
                            'zipcode_4tier_rank',
                            'overall_condition_rank',
                            'heat_type_rank'
                         ]



    pipeline = make_pipeline(
        DictVectorizer(),
        ElasticNet(alpha=0.01, l1_ratio=0.01, max_iter=2000, random_state=1)
    )
    
    target = 'total_value'

    
    df_linear = df[categorical + numerical + [target]]
    df_full_train, df_test = train_test_split(df_linear, test_size=0.2, random_state=1)
    re_test_value = df_test.iloc[10]
    
    y_train = np.log(df_linear.total_value.values + 1)
    df_full_train = df_full_train.reset_index(drop = True)
    
    train_dict = df[categorical + numerical].to_dict(orient='records')
    
    pipeline.fit(train_dict, y_train)
    return re_test_value, pipeline 


In [43]:
def save_model(filename, model):
    with open(filename, 'wb') as f_out:
        pickle.dump(model, f_out)
    print(f'model saved to {filename}')

In [44]:
df = load_data()
test_value, pipeline  = train_model(df)
save_model('model.bin', pipeline)

model saved to model.bin


In [46]:
print(test_value)

land_use_description_rank         2.00
zipcode_4tier_rank                3.00
overall_condition_rank            4.00
heat_type_rank                    6.00
total_num_rooms                   8.00
gross_area                     3278.00
adjusted_tax                   6152.92
bed_rms                           4.00
full_bth                          2.00
kitchens                          1.00
num_parking                       1.00
floor_number                      1.50
building_age                     97.00
remodel_age                       6.00
total_value                  875400.00
Name: 64607, dtype: float64
